# C. elegans Connectome: Loading & Visualization

The C. elegans nervous system has **302 neurons** connected by ~3,000 chemical synapses and ~1,000 gap junctions — the only organism with a completely mapped connectome. This notebook loads and explores this wiring diagram.

In [ ]:
import sys
sys.path.insert(0, "../creatures-core")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from creatures.connectome.openworm import load
from creatures.connectome.types import NeuronType

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load the connectome from the OpenWorm edge list
connectome = load("edge_list")
print(connectome.summary())

## Neuron Type Distribution

C. elegans neurons fall into three functional categories:
- **Sensory** — detect environmental stimuli (touch, chemicals, temperature)
- **Interneurons** — process information and make decisions
- **Motor** — drive muscle contraction for movement

In [ ]:
# Neuron type breakdown
type_counts = {}
for n in connectome.neurons.values():
    type_counts[n.neuron_type.value] = type_counts.get(n.neuron_type.value, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = {"sensory": "#4CAF50", "inter": "#2196F3", "motor": "#FF5722"}
labels = list(type_counts.keys())
sizes = list(type_counts.values())
axes[0].pie(sizes, labels=[f"{l}\n({s})" for l, s in zip(labels, sizes)],
            colors=[colors.get(l, "#999") for l in labels],
            autopct="%1.0f%%", startangle=90, textprops={"fontsize": 12})
axes[0].set_title("Neuron Types", fontsize=14, fontweight="bold")

# Neurotransmitter breakdown
nt_counts = {}
for n in connectome.neurons.values():
    nt = n.neurotransmitter or "unknown"
    nt_counts[nt] = nt_counts.get(nt, 0) + 1

nt_sorted = sorted(nt_counts.items(), key=lambda x: -x[1])
nt_labels, nt_values = zip(*nt_sorted)
nt_colors = {"ACh": "#E91E63", "glutamate": "#9C27B0", "GABA": "#00BCD4",
             "dopamine": "#FF9800", "serotonin": "#CDDC39", "unknown": "#9E9E9E"}
bar_colors = [nt_colors.get(l, "#999") for l in nt_labels]

axes[1].barh(range(len(nt_labels)), nt_values, color=bar_colors)
axes[1].set_yticks(range(len(nt_labels)))
axes[1].set_yticklabels(nt_labels, fontsize=11)
axes[1].set_xlabel("Number of neurons", fontsize=12)
axes[1].set_title("Neurotransmitter Distribution", fontsize=14, fontweight="bold")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Adjacency Matrix

The adjacency matrix shows every connection in the nervous system. Each cell (i, j) represents the number of synapses from neuron i to neuron j. Positive values are excitatory, negative are inhibitory (GABA).

In [ ]:
# Adjacency matrix heatmap
adj = connectome.adjacency_matrix
neuron_ids = connectome.neuron_ids

# Sort neurons by type for clearer visualization
type_order = {NeuronType.SENSORY: 0, NeuronType.INTER: 1, NeuronType.MOTOR: 2}
sorted_indices = sorted(range(len(neuron_ids)),
                        key=lambda i: (type_order.get(connectome.neurons[neuron_ids[i]].neuron_type, 3),
                                       neuron_ids[i]))
sorted_adj = adj[np.ix_(sorted_indices, sorted_indices)]
sorted_ids = [neuron_ids[i] for i in sorted_indices]

# Find boundaries between neuron types
boundaries = []
current_type = connectome.neurons[sorted_ids[0]].neuron_type
for i, nid in enumerate(sorted_ids):
    if connectome.neurons[nid].neuron_type != current_type:
        boundaries.append(i)
        current_type = connectome.neurons[nid].neuron_type

fig, ax = plt.subplots(figsize=(12, 10))
vmax = np.percentile(np.abs(sorted_adj[sorted_adj != 0]), 95)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(sorted_adj, cmap="RdBu_r", norm=norm, aspect="equal", interpolation="nearest")

# Draw type boundaries
for b in boundaries:
    ax.axhline(b - 0.5, color="white", linewidth=1.5, alpha=0.8)
    ax.axvline(b - 0.5, color="white", linewidth=1.5, alpha=0.8)

# Label regions
type_labels = ["Sensory", "Inter", "Motor"]
starts = [0] + boundaries
ends = boundaries + [len(sorted_ids)]
for label, start, end in zip(type_labels, starts, ends):
    mid = (start + end) / 2
    ax.text(-8, mid, label, ha="right", va="center", fontsize=10, fontweight="bold",
            color=colors.get(label.lower(), "black"))

plt.colorbar(im, ax=ax, label="Signed synapse count", shrink=0.8)
ax.set_title(f"C. elegans Connectome ({connectome.n_neurons} neurons, {connectome.n_synapses} synapses)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Postsynaptic neuron")
ax.set_ylabel("Presynaptic neuron")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

print(f"Matrix density: {(sorted_adj != 0).sum() / sorted_adj.size:.1%}")
print(f"Excitatory connections: {(sorted_adj > 0).sum()}")
print(f"Inhibitory connections: {(sorted_adj < 0).sum()}")

## Degree Distribution

How connected is each neuron? Hub neurons with high degree play critical roles in information routing.

In [ ]:
# Degree distribution
in_degrees = {}
out_degrees = {}
for s in connectome.synapses:
    out_degrees[s.pre_id] = out_degrees.get(s.pre_id, 0) + 1
    in_degrees[s.post_id] = in_degrees.get(s.post_id, 0) + 1

total_degrees = {}
for nid in connectome.neuron_ids:
    total_degrees[nid] = in_degrees.get(nid, 0) + out_degrees.get(nid, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
all_deg = list(total_degrees.values())
axes[0].hist(all_deg, bins=30, color="#5C6BC0", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Total degree (in + out)", fontsize=12)
axes[0].set_ylabel("Number of neurons", fontsize=12)
axes[0].set_title("Degree Distribution", fontsize=14, fontweight="bold")
axes[0].axvline(np.mean(all_deg), color="red", linestyle="--", label=f"Mean: {np.mean(all_deg):.0f}")
axes[0].legend()

# Top hub neurons
top_hubs = sorted(total_degrees.items(), key=lambda x: -x[1])[:20]
hub_ids, hub_degs = zip(*top_hubs)
hub_types = [connectome.neurons[nid].neuron_type.value for nid in hub_ids]
hub_colors = [colors.get(t, "#999") for t in hub_types]

axes[1].barh(range(len(hub_ids)), hub_degs, color=hub_colors)
axes[1].set_yticks(range(len(hub_ids)))
axes[1].set_yticklabels(hub_ids, fontsize=9)
axes[1].set_xlabel("Total degree", fontsize=12)
axes[1].set_title("Top 20 Hub Neurons", fontsize=14, fontweight="bold")
axes[1].invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l.capitalize()) for l, c in colors.items()]
axes[1].legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

print(f"Most connected neuron: {top_hubs[0][0]} ({top_hubs[0][1]} connections)")
print(f"Average degree: {np.mean(all_deg):.1f}")
print(f"Median degree: {np.median(all_deg):.1f}")

## Touch Withdrawal Circuit

The best-studied circuit in C. elegans: touch sensory neurons (ALM, PLM) connect through command interneurons (AVA, AVB, AVD, PVC) to motor neurons, enabling the worm to reverse away from touch.

In [ ]:
# Extract and visualize the touch withdrawal circuit
touch_circuit_ids = [
    # Sensory
    "ALML", "ALMR", "PLML", "PLMR", "AVM",
    # Command interneurons
    "AVAL", "AVAR", "AVBL", "AVBR", "AVDL", "AVDR", "PVCL", "PVCR",
    # Key motor neurons (forward/backward)
    "DA1", "DA2", "DA3", "VA1", "VA2", "VA3",
    "DB1", "DB2", "DB3", "VB1", "VB2", "VB3",
]

touch_circuit = connectome.subset(touch_circuit_ids)
print(touch_circuit.summary())
print()

# Visualize as adjacency matrix
touch_adj = touch_circuit.adjacency_matrix
touch_ids = touch_circuit.neuron_ids

fig, ax = plt.subplots(figsize=(10, 8))
vmax = np.max(np.abs(touch_adj[touch_adj != 0])) if (touch_adj != 0).any() else 1
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(touch_adj, cmap="RdBu_r", norm=norm, aspect="equal")

ax.set_xticks(range(len(touch_ids)))
ax.set_yticks(range(len(touch_ids)))
ax.set_xticklabels(touch_ids, rotation=90, fontsize=8)
ax.set_yticklabels(touch_ids, fontsize=8)

# Color labels by neuron type
type_colors_map = {NeuronType.SENSORY: "#4CAF50", NeuronType.INTER: "#2196F3", NeuronType.MOTOR: "#FF5722"}
for i, nid in enumerate(touch_ids):
    nt = touch_circuit.neurons[nid].neuron_type
    color = type_colors_map.get(nt, "black")
    ax.get_yticklabels()[i].set_color(color)
    ax.get_xticklabels()[i].set_color(color)

# Annotate non-zero cells
for i in range(len(touch_ids)):
    for j in range(len(touch_ids)):
        val = touch_adj[i, j]
        if val != 0:
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(val) > vmax * 0.5 else "black")

plt.colorbar(im, ax=ax, label="Signed synapse count", shrink=0.8)
ax.set_title("Touch Withdrawal Circuit", fontsize=14, fontweight="bold")
ax.set_xlabel("Postsynaptic")
ax.set_ylabel("Presynaptic")
plt.tight_layout()
plt.show()

print(f"\nSensory → Interneuron connections:")
for s in touch_circuit.synapses:
    pre = touch_circuit.neurons[s.pre_id]
    post = touch_circuit.neurons[s.post_id]
    if pre.neuron_type == NeuronType.SENSORY and post.neuron_type == NeuronType.INTER:
        print(f"  {s.pre_id} → {s.post_id}: {s.weight:.0f} synapses ({s.synapse_type.value})")

## Brian2 Parameters

Preview of how the connectome translates to Brian2 spiking neural network parameters.

In [ ]:
# Brian2 parameters for the full connectome
b2 = connectome.to_brian2_params()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weight distribution
weights = b2["w"]
axes[0].hist(weights[weights > 0], bins=30, color="#E91E63", alpha=0.7, label="Excitatory")
axes[0].hist(weights[weights < 0], bins=30, color="#00BCD4", alpha=0.7, label="Inhibitory")
axes[0].set_xlabel("Synaptic weight (signed synapse count)", fontsize=12)
axes[0].set_ylabel("Number of connections", fontsize=12)
axes[0].set_title("Weight Distribution", fontsize=14, fontweight="bold")
axes[0].legend()

# Connection density by neuron type
type_names = ["Sensory", "Inter", "Motor"]
type_enums = [NeuronType.SENSORY, NeuronType.INTER, NeuronType.MOTOR]
idx = connectome.neuron_id_to_index
density_matrix = np.zeros((3, 3))

for s in connectome.synapses:
    pre_type = connectome.neurons[s.pre_id].neuron_type
    post_type = connectome.neurons[s.post_id].neuron_type
    if pre_type in type_enums and post_type in type_enums:
        i = type_enums.index(pre_type)
        j = type_enums.index(post_type)
        density_matrix[i, j] += 1

im = axes[1].imshow(density_matrix, cmap="YlOrRd", aspect="equal")
axes[1].set_xticks(range(3))
axes[1].set_yticks(range(3))
axes[1].set_xticklabels(type_names, fontsize=11)
axes[1].set_yticklabels(type_names, fontsize=11)
axes[1].set_xlabel("Postsynaptic type")
axes[1].set_ylabel("Presynaptic type")
axes[1].set_title("Connection Count by Type", fontsize=14, fontweight="bold")

for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f"{int(density_matrix[i,j])}", ha="center", va="center",
                     fontsize=12, fontweight="bold",
                     color="white" if density_matrix[i,j] > density_matrix.max()*0.5 else "black")

plt.colorbar(im, ax=axes[1], label="Number of synapses", shrink=0.8)
plt.tight_layout()
plt.show()

print(f"Total connections: {len(b2['i'])}")
print(f"Excitatory: {(weights > 0).sum()}, Inhibitory: {(weights < 0).sum()}")
print(f"Ready for Brian2: {b2['i'].shape[0]} synapses across {connectome.n_neurons} neurons")